# King County Refactored

The same analysis as [King-County.ipynb](King-County.ipynb), with the cleaning,
feature engineering and modeling moved into the `kc` package.

What stays in a notebook and what does not:

| Stays here | Moved to `kc/` |
|---|---|
| Charts, maps, and the narrative around them | Cleaning rules |
| Trying things out | Feature formulas |
| Reading the results | The model pipeline, scoring, saving |

The test is simple. Everything below reads like a description of the analysis;
nothing below re-implements it. If a rule needs changing, it changes in one file
and every caller follows: this notebook, the training script, the API.

Run `python -m kc.train_amin` to reproduce the final model without opening this file.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

from kc.cleaning_amin import clean, missing_value_report
from kc.data_amin import load_raw
from kc.features_amin import add_features
from kc.modeling_amin import error_table, evaluate, save_model, split_dataset
from kc.pipeline_amin import build_baseline_pipeline, build_model_pipeline

## Data preparation

`load_raw` reads the CSV untouched. `clean` applies every fix from the original
notebook's Data Preparation section:

| Fix | Why |
|---|---|
| drop houses with more than 30 bedrooms | one record lists 33 bedrooms in 1620 sqft |
| rebuild `sqft_basement` | 454 rows hold `?`, so pandas read the column as text |
| fill `view` and `waterfront` with 0 | both are dominated by 0; missing means "no" |
| merge `yr_built`/`yr_renovated` into `last_known_change` | `yr_renovated` is missing or 0 for 20853 rows |

In [ ]:
raw = load_raw()
cleaned = clean(raw)

print(f"raw:     {raw.shape[0]} rows, {raw.shape[1]} columns")
print(f"cleaned: {cleaned.shape[0]} rows, {cleaned.shape[1]} columns")
print(f"missing values before: {raw.isna().sum().sum()}, after: {cleaned.isna().sum().sum()}")

In [ ]:
# Where the missing values were, before cleaning.
missing_value_report(raw)

In [ ]:
# `clean` is idempotent, so running it twice changes nothing. That is what makes it
# safe to call on a request at serving time as well as on the training set.
clean(cleaned).equals(cleaned)

## Exploratory data analysis

This is the part that belongs in a notebook. The charts are here to be looked at,
not to be re-run by a script.

In [ ]:
columns_histogram = [
    "price", "bathrooms", "bedrooms", "floors",
    "grade", "last_known_change", "sqft_living", "sqft_lot",
]
cleaned[columns_histogram].hist(bins=50, figsize=(20, 15))
plt.show()

`price` and `sqft_living` are right-skewed, most houses have one floor and three
bedrooms, and extreme lot sizes flatten the lot histogram.

The expensive houses are outliers but they are not errors. Rare is not the same as
wrong, which is exactly why `clean` removes the 33-bedroom record and keeps the
$7.7M sale.

In [ ]:
numeric = cleaned.select_dtypes(include=["number"])
mask = np.triu(numeric.corr())
plt.figure(figsize=(20, 15))
sns.heatmap(round(numeric.corr(), 2), annot=True, mask=mask, cmap="RdBu_r");

## Feature engineering

`add_features` derives all three columns in one call:

| Column | Meaning |
|---|---|
| `sqft_price` | price per square foot of living area plus lot |
| `center_distance` | km to Bill Gates' estate in Medina, a "centre of wealth" proxy |
| `water_distance` | km to the nearest of the 146 waterfront houses |

`water_distance` is the one worth noting. The original notebook computes it with a
Python loop nested inside another Python loop, 21,596 x 146 iterations, minutes of
runtime. `kc.features_amin.water_distance` does the same arithmetic as a single NumPy
broadcast, and `tests/test_features_amin.py` checks the two agree exactly.

In [ ]:
featured = add_features(cleaned)
featured[["sqft_price", "center_distance", "water_distance"]].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(y="price", x="center_distance", data=featured, s=8, ax=axes[0])
axes[0].set_title("Price vs. distance to the wealth centre")
sns.scatterplot(y="price", x="water_distance", data=featured, s=8, ax=axes[1])
axes[1].set_title("Price vs. distance to the water")
plt.tight_layout()
plt.show()

Both hypotheses hold: prices fall as either distance grows.

In [ ]:
fig = px.scatter_map(
    featured,
    lat="lat", lon="long", hover_name="id",
    hover_data=["sqft_price", "sqft_living", "zipcode", "floors"],
    size="sqft_price", color="center_distance",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7, height=400,
)
fig.update_layout(map_style="open-street-map", margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

## Modeling

`build_model_pipeline` returns a single object that cleans, derives features,
selects columns, expands them to polynomial terms, scales, and fits, in that order,
inside cross-validation folds, so nothing leaks from validation data into training.

Two things the original notebook does differently:

* It builds the waterfront reference list from the **whole** dataset before
  splitting, so a training feature is informed by test rows. `WaterDistance` learns
  it during `fit`, from training rows only.
* It leaves `id` in `X` and only removes it on the polynomial path. Here it is
  excluded once, for every model.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(featured)
print(f"train {X_train.shape[0]} rows | test {X_test.shape[0]} rows")

In [ ]:
# The two baselines from the original notebook.
scores = []
for variables in (["grade"], ["grade", "last_known_change"]):
    baseline = build_baseline_pipeline(variables).fit(X_train, y_train)
    scores.append(evaluate(baseline, X_test, y_test, label=" + ".join(variables)))

pd.DataFrame(scores)[["label", "n_features", "r2", "adjusted_r2", "rmse"]]

`grade` alone explains about 43% of the variance; adding `last_known_change` takes
it to about 48%. That is the number the full model has to beat to be worth its
complexity.

In [ ]:
# The tuned parameters found by `python -m kc.train_amin`, applied directly so this
# notebook does not have to re-run the grid search.
model = build_model_pipeline(alpha=0.01, l1_ratio=0.2).fit(X_train, y_train)

full = evaluate(model, X_test, y_test, label="elasticnet, degree 2")
pd.DataFrame(scores + [full])[["label", "n_features", "r2", "adjusted_r2", "rmse", "mae"]]

19 input columns become 209 polynomial features, and adjusted R^2 rises from 0.48
to roughly 0.84.

Note the RMSE column next to it. "Explains 84% of the variance" and "is typically
wrong by about $143,000" describe the same model; only the second is useful when
deciding what to bid on a house.

In [ ]:
# Which coefficients did the L1 penalty push to zero?
coefficients = model.named_steps["model"].regressor_.coef_
print(f"{(coefficients == 0).sum()} of {coefficients.size} coefficients are exactly zero")

## Error analysis

A single score hides where the model fails. The percentage error does not.

In [ ]:
errors = error_table(model, X_test, y_test)
errors.sort_values("price_difference_percent", ascending=False).head()

In [ ]:
fig = px.scatter_map(
    errors,
    lat="latitude", lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7, height=400,
)
fig.update_layout(map_style="open-street-map", margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

The worst over-predictions are worth looking up individually in the King County
[property lookup](https://localscape.property/#kingcountyassessor/My-Property) by
parcel ID. In the original notebook one of them turned out to be a lot with no
house standing on it any more. The model priced a building that no longer existed.
That is a data problem, not a modeling problem, and no amount of tuning fixes it.

## Save the model

`save_model` writes the whole pipeline, preprocessing included. Anything that loads
it gets the exact transformations the model was trained with, which is what lets the
FastAPI service accept raw house attributes instead of engineered features.

In [ ]:
save_model(model)

## What was done, and what is left

**Steps taken.** Read the original notebook end to end; moved the Data Preparation
section into `kc/cleaning_amin.py` and the Feature Engineering section into
`kc/features_amin.py` as pure functions; wrapped both in scikit-learn transformers so
they fit inside a pipeline and travel with the saved model; reduced the modeling
section to `python -m kc.train_amin`; wrapped the result in a FastAPI service with CRUD
and a `/predict` endpoint, containerised with Docker Compose; and covered the whole
thing with 54 tests.

**Challenges.** Three were real rather than mechanical:

1. *`inplace=True` and positional drops.* The original cleaning cannot be run twice.
   Every function here returns a new frame and `clean(clean(df)) == clean(df)` is a
   test.
2. *The waterfront leak.* Deriving `water_distance` from the full dataset before
   splitting quietly lets test rows shape a training feature. Moving the reference
   set into `fit` was the fix, and it is also what makes predicting for a single
   house possible at all, since one house has no waterfront neighbours of its own.
3. *Serving a pipeline, not a model.* The API's five stored fields are nowhere near
   the nineteen the estimator needs. Putting the preprocessing inside the saved
   object meant `/predict` could take raw attributes and stay in step with training.

**With more time.** The target is worth revisiting: the original text talks about
price per square foot throughout while the code fits total price, so it is worth settling
deliberately. `zipcode` is treated as a number, so the model believes 98104 is
"between" 98103 and 98105; one-hot or target encoding would respect that it is a
label. And `sqft_above + sqft_basement` equals `sqft_living` exactly, so three
perfectly dependent columns go into the polynomial expansion. Dropping one would
lose nothing and help the conditioning.